# GPT-2 Medium/Large Two-Stage WikiText Fine-Tuning + Validation
This notebook fine-tunes `gpt2-medium` and `gpt2-large` for decoder-native next-token language modeling on WikiText.

Flow:
1) optional stage-1 LM fine-tune (e.g., `wikitext-103-raw-v1`)
2) stage-2 LM fine-tune (default `wikitext-2-raw-v1`)
3) evaluate each stage with **strided perplexity** on validation splits

Primary benchmark metric: **perplexity (PPL)** (lower is better).

In [ ]:
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)

In [ ]:
MODELS = ["gpt2-medium", "gpt2-large"]

# Optional stage-1 (set None to skip)
STAGE1_WIKITEXT_CONFIG = None  # e.g. "wikitext-103-raw-v1"
STAGE1_TRAIN_SPLIT = "train"
STAGE1_VAL_SPLIT = "validation"
STAGE1_TRAIN_LIMIT = None
STAGE1_VAL_LIMIT = None

# Stage-2 target dataset
STAGE2_WIKITEXT_CONFIG = "wikitext-2-raw-v1"
STAGE2_TRAIN_SPLIT = "train"
STAGE2_VAL_SPLIT = "validation"
STAGE2_TRAIN_LIMIT = None
STAGE2_VAL_LIMIT = None

# LM/training settings
BLOCK_SIZE = 512
EVAL_STRIDE = 512

STAGE1_EPOCHS = 1.0
STAGE2_EPOCHS = 1.0
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

DEFAULT_BATCH_SIZE = 2
DEFAULT_GRAD_ACCUM = 8
MODEL_BATCHING = {
    "gpt2-medium": {"batch_size": 2, "grad_accum": 8},
    "gpt2-large": {"batch_size": 1, "grad_accum": 16},
}

LOGGING_STEPS = 50
SEED = 123

USE_FP16 = True
USE_BF16 = True
GRADIENT_CHECKPOINTING = True

REPO_ROOT = Path.cwd().resolve()
OUTPUT_ROOT = REPO_ROOT / "checkpoints"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", DEVICE)
print("repo root:", REPO_ROOT)
print("checkpoints root:", OUTPUT_ROOT)

In [ ]:
def now_id():
    return time.strftime("%Y%m%d-%H%M%S")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def bf16_is_available():
    if not torch.cuda.is_available():
        return False
    if not hasattr(torch.cuda, "is_bf16_supported"):
        return False
    return bool(torch.cuda.is_bf16_supported())


def load_wikitext_lines(config_name, split, limit=None):
    ds = load_dataset("wikitext", config_name, split=split)
    n = len(ds) if limit is None else min(int(limit), len(ds))
    if limit is not None:
        ds = ds.select(range(n))

    lines = []
    for txt in ds["text"]:
        txt = str(txt)
        if txt.strip():
            lines.append(txt)

    return lines


def build_lm_blocks(lines, tok, block_size):
    ds = Dataset.from_dict({"text": lines})

    def tokenize_fn(batch):
        return tok(batch["text"], add_special_tokens=False)

    tok_ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])

    def group_texts(batch):
        concat = {k: sum(batch[k], []) for k in batch.keys()}
        total_len = len(concat["input_ids"])
        total_len = (total_len // block_size) * block_size
        out = {
            k: [vals[i : i + block_size] for i in range(0, total_len, block_size)]
            for k, vals in concat.items()
        }
        out["labels"] = [x.copy() for x in out["input_ids"]]
        return out

    lm_ds = tok_ds.map(group_texts, batched=True)
    return lm_ds

In [ ]:
def load_model_and_tokenizer(model_id_or_path):
    tok = AutoTokenizer.from_pretrained(model_id_or_path, use_fast=True)
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_id_or_path, attn_implementation="eager")
    if tok.pad_token_id is not None:
        model.config.pad_token_id = tok.pad_token_id
    model.to(DEVICE)
    model.train()
    return tok, model


def train_stage_lm(
    model,
    tok,
    train_lines,
    stage_name,
    out_dir,
    epochs,
    batch_size,
    grad_accum,
    learning_rate,
    warmup_ratio,
    weight_decay,
    logging_steps,
    seed,
):
    train_ds = build_lm_blocks(train_lines, tok, BLOCK_SIZE)
    if len(train_ds) == 0:
        raise RuntimeError("No LM blocks were created. Increase data size or reduce BLOCK_SIZE.")


    use_fp16 = bool(USE_FP16 and torch.cuda.is_available() and not bf16_is_available())
    use_bf16 = bool(USE_BF16 and bf16_is_available())

    model_ctx = int(getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", 1024)))
    if int(BLOCK_SIZE) > model_ctx:
        raise ValueError(f"BLOCK_SIZE={BLOCK_SIZE} exceeds model context {model_ctx}")

    if GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    args = TrainingArguments(
        output_dir=str(out_dir),
        overwrite_output_dir=True,
        per_device_train_batch_size=int(batch_size),
        gradient_accumulation_steps=int(grad_accum),
        learning_rate=float(learning_rate),
        num_train_epochs=float(epochs),
        warmup_ratio=float(warmup_ratio),
        weight_decay=float(weight_decay),
        evaluation_strategy="no",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=int(logging_steps),
        report_to="none",
        seed=int(seed),
        fp16=use_fp16,
        bf16=use_bf16,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=default_data_collator,
        tokenizer=tok,
    )

    train_metrics = trainer.train().metrics
    trainer.save_model(str(out_dir))
    tok.save_pretrained(str(out_dir))

    out = {k: float(v) for k, v in train_metrics.items() if isinstance(v, (int, float))}
    out["stage"] = stage_name
    out["train_lines"] = int(len(train_lines))
    out["train_blocks"] = int(len(train_ds))
    out["output_dir"] = str(out_dir)

    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return out


@torch.no_grad()
def evaluate_perplexity_strided(model, tok, val_lines, stride):
    model.eval()

    if len(val_lines) == 0:
        raise RuntimeError("Validation text is empty")

    text = "\n\n".join(val_lines)
    enc = tok(text, return_tensors="pt", add_special_tokens=False)
    input_ids = enc["input_ids"].to(DEVICE)

    seq_len = int(input_ids.size(1))
    max_len = int(getattr(model.config, "n_positions", getattr(model.config, "max_position_embeddings", 1024)))
    stride = int(stride)
    if stride <= 0:
        raise ValueError(f"EVAL_STRIDE must be > 0, got {stride}")

    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_len, seq_len)
        trg_len = end - prev_end

        window_ids = input_ids[:, begin:end]
        target_ids = window_ids.clone()
        target_ids[:, :-trg_len] = -100

        out = model(window_ids, labels=target_ids)
        neg_log_likelihood = float(out.loss.item()) * trg_len

        nll_sum += neg_log_likelihood
        n_tokens += trg_len
        prev_end = end

        if end >= seq_len:
            break

    avg_nll = nll_sum / max(1, n_tokens)
    ppl = float(math.exp(avg_nll)) if avg_nll < 20 else float("inf")

    return {
        "n_tokens": int(n_tokens),
        "avg_nll": float(avg_nll),
        "ppl": float(ppl),
    }

In [ ]:
seed_everything(SEED)

stage2_train_lines = load_wikitext_lines(STAGE2_WIKITEXT_CONFIG, STAGE2_TRAIN_SPLIT, STAGE2_TRAIN_LIMIT)
stage2_val_lines = load_wikitext_lines(STAGE2_WIKITEXT_CONFIG, STAGE2_VAL_SPLIT, STAGE2_VAL_LIMIT)

if STAGE1_WIKITEXT_CONFIG is None:
    stage1_train_lines = []
    stage1_val_lines = []
else:
    stage1_train_lines = load_wikitext_lines(STAGE1_WIKITEXT_CONFIG, STAGE1_TRAIN_SPLIT, STAGE1_TRAIN_LIMIT)
    stage1_val_lines = load_wikitext_lines(STAGE1_WIKITEXT_CONFIG, STAGE1_VAL_SPLIT, STAGE1_VAL_LIMIT)

print("stage2_train_lines:", len(stage2_train_lines))
print("stage2_val_lines:", len(stage2_val_lines))
print("stage1_train_lines:", len(stage1_train_lines))
print("stage1_val_lines:", len(stage1_val_lines))

if len(stage2_train_lines) == 0 or len(stage2_val_lines) == 0:
    raise RuntimeError("Stage-2 WikiText lines are empty")

In [ ]:
run_id = now_id()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

all_metrics = []
train_logs = []
model_artifacts = []

for model_name in MODELS:
    print("=" * 80)
    print("model:", model_name)

    batch_cfg = MODEL_BATCHING.get(model_name, {})
    train_batch = int(batch_cfg.get("batch_size", DEFAULT_BATCH_SIZE))
    grad_accum = int(batch_cfg.get("grad_accum", DEFAULT_GRAD_ACCUM))

    tok, model = load_model_and_tokenizer(model_name)

    base_stage2 = evaluate_perplexity_strided(model, tok, stage2_val_lines, EVAL_STRIDE)
    all_metrics.append(
        {
            "model": model_name,
            "stage": "base",
            "dataset": f"{STAGE2_WIKITEXT_CONFIG}_validation",
            **base_stage2,
        }
    )

    if len(stage1_val_lines) > 0:
        base_stage1 = evaluate_perplexity_strided(model, tok, stage1_val_lines, EVAL_STRIDE)
        all_metrics.append(
            {
                "model": model_name,
                "stage": "base",
                "dataset": f"{STAGE1_WIKITEXT_CONFIG}_validation",
                **base_stage1,
            }
        )

    if STAGE1_WIKITEXT_CONFIG is not None and len(stage1_train_lines) > 0:
        stage1_dir = OUTPUT_ROOT / f"{run_id}_{model_name}_stage1_{STAGE1_WIKITEXT_CONFIG}".replace("/", "-")
        log1 = train_stage_lm(
            model=model,
            tok=tok,
            train_lines=stage1_train_lines,
            stage_name="stage1",
            out_dir=stage1_dir,
            epochs=STAGE1_EPOCHS,
            batch_size=train_batch,
            grad_accum=grad_accum,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_steps=LOGGING_STEPS,
            seed=SEED,
        )
        train_logs.append({"model": model_name, **log1})

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        tok, model = load_model_and_tokenizer(str(stage1_dir))

        s1_stage2 = evaluate_perplexity_strided(model, tok, stage2_val_lines, EVAL_STRIDE)
        all_metrics.append(
            {
                "model": model_name,
                "stage": "after_stage1",
                "dataset": f"{STAGE2_WIKITEXT_CONFIG}_validation",
                **s1_stage2,
            }
        )

        if len(stage1_val_lines) > 0:
            s1_stage1 = evaluate_perplexity_strided(model, tok, stage1_val_lines, EVAL_STRIDE)
            all_metrics.append(
                {
                    "model": model_name,
                    "stage": "after_stage1",
                    "dataset": f"{STAGE1_WIKITEXT_CONFIG}_validation",
                    **s1_stage1,
                }
            )

    stage2_dir = OUTPUT_ROOT / f"{run_id}_{model_name}_stage2_{STAGE2_WIKITEXT_CONFIG}".replace("/", "-")
    log2 = train_stage_lm(
        model=model,
        tok=tok,
        train_lines=stage2_train_lines,
        stage_name="stage2",
        out_dir=stage2_dir,
        epochs=STAGE2_EPOCHS,
        batch_size=train_batch,
        grad_accum=grad_accum,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        logging_steps=LOGGING_STEPS,
        seed=SEED,
    )
    train_logs.append({"model": model_name, **log2})

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    tok, model = load_model_and_tokenizer(str(stage2_dir))

    final_stage2 = evaluate_perplexity_strided(model, tok, stage2_val_lines, EVAL_STRIDE)
    all_metrics.append(
        {
            "model": model_name,
            "stage": "after_stage2",
            "dataset": f"{STAGE2_WIKITEXT_CONFIG}_validation",
            **final_stage2,
        }
    )

    if len(stage1_val_lines) > 0:
        final_stage1 = evaluate_perplexity_strided(model, tok, stage1_val_lines, EVAL_STRIDE)
        all_metrics.append(
            {
                "model": model_name,
                "stage": "after_stage2",
                "dataset": f"{STAGE1_WIKITEXT_CONFIG}_validation",
                **final_stage1,
            }
        )

    model_artifacts.append(
        {
            "model": model_name,
            "stage2_checkpoint": str(stage2_dir),
        }
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

metrics_df = pd.DataFrame(all_metrics)
train_df = pd.DataFrame(train_logs)

metrics_df

In [ ]:
if len(metrics_df) > 0:
    display(metrics_df.sort_values(["model", "dataset", "stage"]).reset_index(drop=True))

if len(metrics_df) > 0:
    pivot_ppl = metrics_df.pivot_table(index=["model", "stage"], columns="dataset", values="ppl")
    display(pivot_ppl)

if len(train_df) > 0:
    display(train_df.sort_values(["model", "stage"]).reset_index(drop=True))

if len(model_artifacts) > 0:
    art_df = pd.DataFrame(model_artifacts)
    display(art_df)

summary_dir = OUTPUT_ROOT / f"{run_id}_wikitext_lm_summary"
summary_dir.mkdir(parents=True, exist_ok=True)

if len(metrics_df) > 0:
    metrics_df.to_csv(summary_dir / "metrics.csv", index=False)
if len(train_df) > 0:
    train_df.to_csv(summary_dir / "train_logs.csv", index=False)
if len(model_artifacts) > 0:
    pd.DataFrame(model_artifacts).to_csv(summary_dir / "artifacts.csv", index=False)

with (summary_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "models": MODELS,
            "stage1_wikitext_config": STAGE1_WIKITEXT_CONFIG,
            "stage2_wikitext_config": STAGE2_WIKITEXT_CONFIG,
            "block_size": BLOCK_SIZE,
            "eval_stride": EVAL_STRIDE,
            "seed": SEED,
        },
        f,
        indent=2,
    )

print("saved summary to:", summary_dir)